<a href="https://colab.research.google.com/github/AtifAli05/AtifAli05/blob/main/interviewTask2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = "/content/drive/MyDrive/interview_docs/AAOIFI/AAOIFI Auditing Standards (1-5).pdf"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import pdfplumber
import pytesseract
from pdf2image import convert_from_path

def extract_text_multilingual(pdf_path):
    text = ""

    # Try normal PDF extraction
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            raw = page.extract_text()
            if raw:
                text += raw + "\n"
            else:
                # OCR fallback
                img = convert_from_path(pdf_path, first_page=page.page_number, last_page=page.page_number)[0]
                ocr_text = pytesseract.image_to_string(img, lang="eng+ara+urd")
                text += ocr_text + "\n"

    return text

full_text = extract_text_multilingual(BASE_DIR)
len(full_text)


160508

In [4]:
def chunk_text(text, chunk_size=500, overlap=50):
    chunks = []
    words = text.split()

    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)

    return chunks

chunks = chunk_text(full_text)
len(chunks)


36

In [5]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

embeddings = model.encode(chunks, convert_to_numpy=True)
embeddings.shape


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

(36, 384)

In [7]:
import faiss

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print("Indexed vectors:", index.ntotal)


Indexed vectors: 36


In [13]:
GDDFGFDGFDGFDGFDGDGFDG

In [14]:
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
import torch

device = "cpu"

model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model_seq = AutoModelForSeq2SeqLM.from_pretrained(model_name)

gen = pipeline(
    "text2text-generation",
    model=model_seq,
    tokenizer=tokenizer,
    device_map=None
)


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Device set to use cpu


In [22]:
import requests, os

API_KEY = os.getenv("groq_api_key")  # set your Groq API key in Colab

def call_groq(prompt: str) -> str:
    url = "https://api.groq.ai/v1/generate"
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }
    payload = {
        "prompt": prompt,
        "max_tokens": 512
    }
    resp = requests.post(url, json=payload, headers=headers)
    resp.raise_for_status()
    return resp.json()["text"]


In [19]:
import requests, os

API_KEY = os.getenv("GROK_API_KEY")

def call_grok(prompt: str) -> str:
    url = "https://api.grok.ai/v1/generate"
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }
    payload = {
        "prompt": prompt,
        "max_tokens": 512
    }
    resp = requests.post(url, json=payload, headers=headers)
    resp.raise_for_status()
    return resp.json()["text"]


In [23]:
import numpy as np
from typing import List, Dict

conversation_memory: List[Dict[str, str]] = []

def retrieve_topk(query: str, k: int = 5):
    # use your FAISS + embeddings for local retrieval
    q_emb = model.encode([query], convert_to_numpy=True)
    scores, idx = index.search(q_emb, k)
    return [{"chunk": chunks[i], "score": float(s), "idx": int(i)}
            for i, s in zip(idx[0], scores[0])]

def build_prompt(query: str, retrieved: List[dict], history: List[dict], max_context_chars: int = 2500) -> str:
    ctx = ""
    for r in retrieved:
        ctx += r["chunk"].strip() + "\n---\n"
        if len(ctx) > max_context_chars:
            break
    history_text = ""
    for ex in history[-6:]:
        if "user" in ex:
            history_text += "User: " + ex["user"].strip() + "\n"
        if "assistant" in ex:
            history_text += "Assistant: " + ex["assistant"].strip() + "\n"
    prompt = (
        "You are an expert assistant. Use the CONTEXT below to answer the USER query.\n\n"
        f"CONTEXT:\n{ctx}\n"
        f"HISTORY:\n{history_text}\n"
        f"USER QUERY:\n{query.strip()}\n\n"
        "TASK: Provide a concise, accurate answer using the CONTEXT. "
        "If the user asked for a prediction/forecast, label the output at the top as 'AI Predicted Response'. "
        "Keep answer length < 300 words. If the context doesn't contain the information, say you couldn't find it and provide next steps.\n\n"
        "Answer:\n"
    )
    return prompt

def generate_rag_answer(query: str, k: int = 5, add_to_memory: bool = True):
    retrieved = retrieve_topk(query, k)
    prompt = build_prompt(query, retrieved, conversation_memory)
    # generate using Groq LLM because  Grok api is apid
    out = call_groq(prompt)
    if add_to_memory:
        conversation_memory.append({"user": query, "assistant": out})
    return {"answer": out, "retrieved": retrieved, "prompt": prompt}


In [25]:
query = "Explain the terms of audit engagement in AAOIFI standards"
res = generate_rag_answer(query)
print(res["answer"])


In [ ]:
import pickle, faiss
with open("rag_chunks.pkl", "wb") as f:
    pickle.dump(chunks, f)

faiss.write_index(index, "rag_index.faiss")
with open("rag_chunks.pkl", "rb") as f:
    chunks = pickle.load(f)

index = faiss.read_index("rag_index.faiss")


In [ ]:
import pandas as pd

BASE = "/content/drive/MyDrive/adventureworks/"

header = pd.read_csv(BASE + "SalesOrderHeader.csv")
detail = pd.read_csv(BASE + "SalesOrderDetail.csv")
product = pd.read_csv(BASE + "Product.csv")
subcategory = pd.read_csv(BASE + "ProductSubcategory.csv")
category = pd.read_csv(BASE + "ProductCategory.csv")

header.head(), detail.head()
df = (detail
      .merge(product, on="ProductID")
      .merge(subcategory, on="ProductSubcategoryID", how="left")
      .merge(category, on="ProductCategoryID", how="left")
      .merge(header[["SalesOrderID", "OrderDate"]], on="SalesOrderID"))

df["OrderDate"] = pd.to_datetime(df["OrderDate"])
